# Hasenzagl 2020 Results

This notebook loads the saved `hasenzagl_2020` results and presents the main common components and the observable fit.

### Initialization

The notebook looks first for a model-specific output file and falls back to the legacy `res_iis_HZ.jld2` file already in this repo.

In [1]:
using Dates, FileIO, JLD2, PlotlyJS, Statistics
pltjs = PlotlyJS

include("./code/Metropolis-Within-Gibbs/MetropolisWithinGibbs.jl")
using Main.MetropolisWithinGibbs

candidate_files = [
    "res_hasenzagl_2020_iis.jld2",
    # "res_iis_HZ.jld2",
]

available_files = filter(isfile, candidate_files)
isempty(available_files) && error("No Hasenzagl result file found. Checked: $(join(candidate_files, ", ")).")
result_file = first(available_files)

println("Using result file: ", result_file)

┌ Warning: Kaledio is not available on this system. Julia will be unable to produce any plots.
└ @ PlotlyBase C:\Users\wrc938\.julia\packages\PlotlyBase\NxSlF\src\kaleido.jl:58


Using result file: res_hasenzagl_2020_iis.jld2


In [2]:
res = load(joinpath(pwd(), result_file))

nDraws = res["nDraws"]
burnin = res["burnin"]
sigma_y = vec(res["σʸ"])
date = res["date"]
data = res["data"] .* sigma_y'
distr_alpha = res["distr_α"]
chain_theta = res["chain_θ_bound"]
mnemonic = vec(res["MNEMONIC"])

draw_idx = axes(distr_alpha, 3)
theta_keep = (burnin[2] + 1):nDraws[2]
n_keep = length(draw_idx)
tt = size(distr_alpha, 2)
n_obs = size(data, 2)

titles = [
    "Real GDP",
    "Employment",
    "Unemployment",
    "Oil",
    "Headline CPI",
    "Core CPI",
    "UoM 1Y Expectations",
    "SPF 1Y Expectations",
]

display((n_obs=n_obs, tt=tt, kept_draws=n_keep, start=first(date), finish=last(date)))


(n_obs = 8, tt = 174, kept_draws = 20000, start = Date("1984-01-01"), finish = Date("2025-04-01"))

### Decompose The Hasenzagl State Space

This follows the same indexing already used in the existing repo notebooks for the legacy Hasenzagl results file.

In [3]:
ind_trends = vcat(collect(9:3:size(distr_alpha, 1))[1:4], collect(25:3:size(distr_alpha, 1)))
ind_cycles = vcat(collect(7:3:size(distr_alpha, 1))[1:4], 19, 21, collect(23:3:size(distr_alpha, 1)))

z_pc_1 = vcat(ones(1, n_keep), chain_theta[1:7, theta_keep])
z_pc_2 = vcat(zeros(1, n_keep), chain_theta[8:14, theta_keep])
z_pc_3 = vcat(zeros(6, n_keep), chain_theta[15:16, theta_keep])

z_ep_1 = vcat(zeros(3, n_keep), ones(1, n_keep), chain_theta[17:20, theta_keep])
z_ep_2 = vcat(zeros(3, n_keep), zeros(1, n_keep), chain_theta[21:24, theta_keep])

z_trend = vcat(zeros(4, n_keep), (1.0 ./ sigma_y[end-3:end]) .* ones(4, n_keep))

pc_draws = zeros(n_obs, tt, n_keep)
ep_draws = zeros(n_obs, tt, n_keep)
trend_draws = zeros(n_obs, tt, n_keep)

for (k, alpha_idx) in enumerate(draw_idx)
    pc_draws[:, :, k] = (z_pc_1[:, k] .* distr_alpha[1, :, alpha_idx]') .+
                        (z_pc_2[:, k] .* distr_alpha[2, :, alpha_idx]') .+
                        (z_pc_3[:, k] .* distr_alpha[3, :, alpha_idx]')
    ep_draws[:, :, k] = (z_ep_1[:, k] .* distr_alpha[4, :, alpha_idx]') .+
                        (z_ep_2[:, k] .* distr_alpha[5, :, alpha_idx]')
    trend_draws[:, :, k] = z_trend[:, k] .* distr_alpha[6, :, alpha_idx]'
end

idio_cycle_draws = distr_alpha[ind_cycles, :, :]
idio_trend_draws = distr_alpha[ind_trends, :, :]

function summarize_draws(draws::AbstractArray{<:Real,3}, scales::AbstractVector{<:Real})
    n_var, tt, n_keep = size(draws)
    med = zeros(tt, n_var)
    q05 = zeros(tt, n_var)
    q16 = zeros(tt, n_var)
    q84 = zeros(tt, n_var)
    q95 = zeros(tt, n_var)

    for i in 1:n_var, t in 1:tt
        x = vec(draws[i, t, :]) .* scales[i]
        med[t, i] = quantile(x, 0.50)
        q05[t, i] = quantile(x, 0.05)
        q16[t, i] = quantile(x, 0.16)
        q84[t, i] = quantile(x, 0.84)
        q95[t, i] = quantile(x, 0.95)
    end

    return (; med, q05, q16, q84, q95)
end

function expand_sparse_summary(summary_raw, obs_idx, n_obs)
    med = zeros(size(summary_raw.med, 1), n_obs)
    q05 = zeros(size(summary_raw.q05, 1), n_obs)
    q16 = zeros(size(summary_raw.q16, 1), n_obs)
    q84 = zeros(size(summary_raw.q84, 1), n_obs)
    q95 = zeros(size(summary_raw.q95, 1), n_obs)

    for (j, idx) in enumerate(obs_idx)
        med[:, idx] = summary_raw.med[:, j]
        q05[:, idx] = summary_raw.q05[:, j]
        q16[:, idx] = summary_raw.q16[:, j]
        q84[:, idx] = summary_raw.q84[:, j]
        q95[:, idx] = summary_raw.q95[:, j]
    end

    return (; med, q05, q16, q84, q95)
end

pc = summarize_draws(pc_draws, sigma_y)
ep = summarize_draws(ep_draws, sigma_y)
trend = summarize_draws(trend_draws, sigma_y)
idio_cycle = summarize_draws(idio_cycle_draws, sigma_y)
idio_trend_raw = summarize_draws(idio_trend_draws, vcat(sigma_y[1:4], sigma_y[7:8]))
idio_trend = expand_sparse_summary(idio_trend_raw, [1, 2, 3, 4, 7, 8], n_obs)

fitted_med = pc.med + ep.med + trend.med + idio_cycle.med + idio_trend.med

function pick_series(summary, idx)
    return (
        med = summary.med[:, idx],
        q05 = summary.q05[:, idx],
        q16 = summary.q16[:, idx],
        q84 = summary.q84[:, idx],
        q95 = summary.q95[:, idx],
    )
end


pick_series (generic function with 1 method)

### Presentation Figures

In [4]:
common_color  = "#2A6EA6"
energy_color  = "#DB4437"
trend_color   = "#3D9970"
idio_color    = "#9467BD"
fitted_color  = "#666666"
actual_color  = "#111111"

common_band68 = "rgba(42, 110, 166, 0.30)"
common_band90 = "rgba(42, 110, 166, 0.15)"
energy_band68 = "rgba(219, 68, 55, 0.30)"
energy_band90 = "rgba(219, 68, 55, 0.15)"
trend_band68  = "rgba(61, 153, 112, 0.30)"
trend_band90  = "rgba(61, 153, 112, 0.15)"

function band_traces(x, summary, name, color, band68, band90)
    return [
        pltjs.scatter(x=x, y=summary.q05, mode="lines", line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
        pltjs.scatter(x=x, y=summary.q95, mode="lines", line=attr(color="rgba(0,0,0,0)"), fill="tonexty", fillcolor=band90, hoverinfo="skip", showlegend=false),
        pltjs.scatter(x=x, y=summary.q16, mode="lines", line=attr(color="rgba(0,0,0,0)"), hoverinfo="skip", showlegend=false),
        pltjs.scatter(x=x, y=summary.q84, mode="lines", line=attr(color="rgba(0,0,0,0)"), fill="tonexty", fillcolor=band68, hoverinfo="skip", showlegend=false),
        pltjs.scatter(x=x, y=summary.med, mode="lines", name=name, line=attr(color=color, width=2.5)),
    ]
end

function base_layout(title_text; width=500, height=420)
    return pltjs.Layout(
        title=title_text,
        titlefont_size=13,
        template="plotly_white",
        hovermode="x unified",
        xaxis=attr(showgrid=true, linecolor="black", mirror=true, nticks=10, tickangle=-45),
        yaxis=attr(showgrid=true, linecolor="black", mirror=true),
        margin=attr(l=55, r=20, t=60, b=45),
        width=width,
        height=height,
    )
end

# --- Key common components ---
fig_pc = pltjs.plot(band_traces(date, pick_series(pc, 1),  "Common price cycle",         common_color, common_band68, common_band90), base_layout("Output common price cycle"))
fig_ep = pltjs.plot(band_traces(date, pick_series(ep, 5),  "Energy-price cycle",          energy_color, energy_band68, energy_band90), base_layout("Headline inflation energy-price cycle"))
fig_tr = pltjs.plot(band_traces(date, pick_series(trend, 5), "Inflation trend",           trend_color,  trend_band68,  trend_band90),  base_layout("Inflation trend"))

plt_common = [fig_pc fig_ep fig_tr]
plt_common.plot.layout["legend"] = attr(orientation="h", y=-0.12, x=0.0, font=attr(size=11))
display(plt_common)

# --- Price block decomposition ---
price_figs = Vector{Any}(undef, 2)
for (col, idx) in enumerate([5, 6])
    traces = [
        pltjs.scatter(x=date, y=data[:, idx],        name="Actual",              line=attr(color=actual_color,  width=2.5),             showlegend=col==1),
        pltjs.scatter(x=date, y=fitted_med[:, idx],  name="Fitted",              line=attr(color=fitted_color,  width=2.5, dash="dash"), showlegend=col==1),
        pltjs.scatter(x=date, y=pc.med[:, idx],      name="Common price cycle",  line=attr(color=common_color,  width=2.0),             showlegend=col==1),
        pltjs.scatter(x=date, y=ep.med[:, idx],      name="Energy-price cycle",  line=attr(color=energy_color,  width=2.0),             showlegend=col==1),
        pltjs.scatter(x=date, y=trend.med[:, idx],   name="Trend",               line=attr(color=trend_color,   width=2.0),             showlegend=col==1),
        pltjs.scatter(x=date, y=idio_cycle.med[:, idx], name="Idiosyncratic cycle", line=attr(color=idio_color, width=2.0),             showlegend=col==1),
    ]
    price_figs[col] = pltjs.plot(traces, base_layout(titles[idx]))
end

plt_prices = [price_figs[1] price_figs[2]]
plt_prices.plot.layout["legend"] = attr(orientation="h", y=-0.12, x=0.0, font=attr(size=11))
display(plt_prices)

# --- Observable fit ---
fit_figs = Vector{Any}(undef, n_obs)
for idx in 1:n_obs
    traces = [
        pltjs.scatter(x=date, y=data[:, idx],       name="Actual",       line=attr(color=actual_color, width=2.25),             showlegend=idx==1),
        pltjs.scatter(x=date, y=fitted_med[:, idx], name="Median fitted", line=attr(color=common_color, width=2.25, dash="dash"), showlegend=idx==1),
    ]
    fit_figs[idx] = pltjs.plot(traces, base_layout(titles[idx], width=380, height=320))
end

plt_fit = [fit_figs[1] fit_figs[2] fit_figs[3] fit_figs[4]; fit_figs[5] fit_figs[6] fit_figs[7] fit_figs[8]]
plt_fit.plot.layout["legend"] = attr(orientation="h", y=-0.04, x=0.0, font=attr(size=11))
display(plt_fit)

data: [
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields fill, fillcolor, hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields fill, fillcolor, hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, type, x, xaxis, y, and yaxis",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields fill, fillcolor, hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields fill, fillcolor, hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, type, x, xaxis, y, and yaxis",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields fill, fillcolor, hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields fill, fillcolor, hoverinfo, line, mode, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, mode, name, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, legend, margin, xaxis1, xaxis2, xaxis3, yaxis1, yaxis2, and yaxis3"

data: [
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, legend, margin, xaxis1, xaxis2, yaxis1, and yaxis2"

data: [
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis",
  "scatter with fields line, name, showlegend, type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, legend, margin, xaxis1, xaxis2, xaxis3, xaxis4, xaxis5, xaxis6, xaxis7, xaxis8, yaxis1, yaxis2, yaxis3, yaxis4, yaxis5, yaxis6, yaxis7, and yaxis8"